NYC 311 Data Ingestion (Optimization Summary)

Data partitioned into 10-day chunks (90-day window) to reduce API load and improve reliability
Each chunk retrieved in 20K record batches using unique_key-based pagination to ensure ordering and avoid duplicates
Query design enforces deterministic sorting and controlled request sizes, mitigating API rate limits and timeouts

Limitation

Intermediate results stored in memory before aggregation, leading to higher RAM usage

Future Improvements

Stream-based processing (fetch → transform → store)
Parquet storage on S3 for scalability
Checkpointing for fault tolerance
Controlled parallel ingestion for improved throughput

In [1]:
from sodapy import Socrata
import pandas as pd
from datetime import datetime, timedelta
import time

client = Socrata("data.cityofnewyork.us", None,timeout = 60)

#cutoff = (datetime.now() - timedelta(days=10)).strftime("%Y-%m-%dT%H:%M:%S")
#current date - 90 days is cutoff

batch_size=20000
all_dfs = []

start_date = (datetime.now() - timedelta(days=90))

start_time = time.time()

for i in range(9):

    chunk_start = start_date + timedelta(days=i*10)
    chunk_end   = chunk_start + timedelta(days=10)

    print(f"Loading Batch :{i} of Data -> From {(str(chunk_start))} to {str(chunk_end)}")
    last_key = 0
    while True:               
        results = client.get(
            "erm2-nwe9",
            where = (f"""created_date >= '{chunk_start.strftime("%Y-%m-%dT%H:%M:%S")}' 
            AND created_date < '{chunk_end.strftime("%Y-%m-%dT%H:%M:%S")}'
            AND unique_key > '{last_key}' """ ),
            order = "unique_key ASC",
            limit = batch_size
        )
    
        if not results:
            break
        
        df_batch = pd.DataFrame(results)
        all_dfs.append(df_batch)

        last_key = df_batch["unique_key"].astype(int).max()
        print("\t",len(df_batch),"Loaded , Last Key : ", last_key)

start_time2= time.time()
dfs = pd.concat(all_dfs, ignore_index=True)
end_time2= time.time()
end_time = time.time()
print("Overall Time",end_time - start_time)
print("Concat Time",end_time2 - start_time2)
print(df_batch.shape)
print(len(all_dfs))
print(dfs.shape)

Loading Batch :0 of Data -> From 2026-03-21 03:14:51.979632 to 2026-03-31 03:14:51.979632
	 20000 Loaded , Last Key :  68423114
	 20000 Loaded , Last Key :  68444299
	 20000 Loaded , Last Key :  68465518
	 20000 Loaded , Last Key :  68486370
	 20000 Loaded , Last Key :  68507071
	 6138 Loaded , Last Key :  68592468
Loading Batch :1 of Data -> From 2026-03-31 03:14:51.979632 to 2026-04-10 03:14:51.979632
	 20000 Loaded , Last Key :  68534776
	 20000 Loaded , Last Key :  68555983
	 20000 Loaded , Last Key :  68576571
	 20000 Loaded , Last Key :  68597951
	 19379 Loaded , Last Key :  68745181
Loading Batch :2 of Data -> From 2026-04-10 03:14:51.979632 to 2026-04-20 03:14:51.979632
	 20000 Loaded , Last Key :  68639430
	 20000 Loaded , Last Key :  68660618
	 20000 Loaded , Last Key :  68682023
	 20000 Loaded , Last Key :  68703366
	 20000 Loaded , Last Key :  68724129
	 2555 Loaded , Last Key :  68791928
Loading Batch :3 of Data -> From 2026-04-20 03:14:51.979632 to 2026-04-30 03:14:51.979

In [4]:
import pyarrow 
import boto3
import s3fs

In [6]:
for r in all_dfs :
    print(r.head(1))
    print(r.shape)
    break

  unique_key             created_date              closed_date agency  \
0   68402018  2026-03-21T09:35:00.000  2026-03-24T09:20:00.000    DEP   

                              agency_name complaint_type  \
0  Department of Environmental Protection   Water System   

                                       descriptor descriptor_2 incident_zip  \
0  Possible Water Main Break (Use Comments) (WA1)          WA1        11691   

        incident_address  ... location_type landmark due_date vehicle_type  \
0  217 BEACH   44 STREET  ...           NaN      NaN      NaN          NaN   

  taxi_pick_up_location bridge_highway_name bridge_highway_segment  \
0                   NaN                 NaN                    NaN   

  bridge_highway_direction road_ramp taxi_company_borough  
0                      NaN       NaN                  NaN  

[1 rows x 44 columns]
(20000, 44)


In [8]:
pip install --upgrade pyarrow pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(dfs)

pq.write_table(table, "raw_data.parquet")